# Session 3: coding agents and independent scientific verification

Work through four scientific gates on one broken analysis function: **1. Inspect & plan $\rightarrow$ 2. Derive expectations & checkpoint tests $\rightarrow$ 3. Bounded repair $\rightarrow$ 4. Independent verification**.

Open your prepared workspace from [coding-agent setup](../book/agent-tools.md) in **Jupyter AI on CPD** (recommended zero-install default), **Cline + CPD Qwen**, **Claude Code**, or follow the **manual route** without an AI agent. Select your tab once in the book (or in each Markdown cell in JupyterLab); keep the terminal for setup, `pytest` and Git.

You own the physical model, expected answers and acceptance decision, finishing with a verified commit and concise `AI_USE.md` record.


## Know what your agent can do

A coding agent connects a **model** (which proposes next steps) to a **client** that runs file and terminal **tools** under your **context and permissions**. That loop can diagnose tracebacks quickly, yet it can also solve the wrong scientific problem or alter tests to make broken code pass. A chat message claiming "tests pass" is never an observed test run.

![Diagram: Locate the client, model and tool execution](../book/diagrams/agent-architecture.svg)

**Separate where the model thinks from where tools act.** In the Cline route, Qwen runs on CPD while file edits and shell commands execute on your laptop ([editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/agent-architecture.dot)).

:::{important} Three distinct boundaries
- **OS/container sandbox:** an enforced process boundary blocking writes outside the workspace or shell network access.
- **Approval mode:** client rules deciding which reads, edits or commands pause for your consent.
- **Git checkpoints:** snapshots for inspecting diffs and rolling back unwanted edits (a branch, folder, `.venv` or prompt instruction is *not* an OS sandbox).
:::

Open `outputs/agent-practice` in your chosen route ([setup reference](../book/agent-tools.md)) and confirm:

- Which client and model are displayed, and on which host do inference versus terminal commands execute?
- Which workspace folder is open, and what approval/sandbox settings apply (keeping credentials and unpublished research data out of prompts)?
- How do you view a proposed diff, interrupt an active turn, and roll back an unwanted edit?


## Supply context and review a plan

![Diagram: Keep scientific expectations under human control](../book/diagrams/agent-review-loop.svg)

**Gate 1 (Plan) and Gate 2 (Tests checkpoint) precede any repair.** You define the scientific contract and lock the agreed test suite before `weighted_mean.py` is edited ([editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/agent-review-loop.dot)).

Inspect `TASK.md`, `weighted_mean.py` and `test_weighted_mean.py` inside `outputs/agent-practice`, then run `python -m pytest -q` in that folder's terminal to record the baseline failures (the starter ignores `errors`). Send the shared inspection prompt below using your tab—or, on the manual route, write a 4-bullet diagnosis and plan before touching code.

::::{tab-set}
:::{tab-item} Jupyter AI on CPD
:sync: agent-jupyterai

In CPD JupyterLab, open **Jupyter Chat → New chat → OpenCode**, confirm `cpd/qwen3.8` and **plan** mode, and attach or name the three files in `outputs/agent-practice`. Choose **Allow once** for scoped file reads and **Reject** for any write.

![Jupyter AI on CPD inspecting the named practice files in OpenCode plan mode](../book/screenshots/jupyter-ai-plan.svg)

**Jupyter AI plan:** **1** names the practice files, **2** shows scoped reads and the plan, and **3** confirms OpenCode, CPD Qwen and plan mode ([full-size image](../book/screenshots/jupyter-ai-plan.svg)).
:::

:::{tab-item} Cline + CPD Qwen
:sync: agent-cline

Open `outputs/agent-practice` in VS Code's **Cline** panel, select **Plan** with **Auto-approve: None**, and confirm `qwen3-8-27b-mtp-ud-q4-k-xl`. Send the prompt below and check both the formula and file paths in its returned plan.

![Cline GUI with the CPD Qwen inspection plan, inverse-variance formula and a proposed test path that needs correction](../book/screenshots/cline-editor-plan.svg)

**Cline plan:** **1** confirms Plan and Auto-approve: None, **2** states inverse-variance weighting, and **3** proposes the wrong `exercises/test_weighted_mean.py` path that your review must catch ([full-size image](../book/screenshots/cline-editor-plan.svg)).
:::

:::{tab-item} Claude Code
:sync: agent-claude

Open `outputs/agent-practice` in VS Code's **Claude Code** panel, select **Plan** beside the composer, and send the inspection prompt. If an **Exit plan mode** dialog appears early, pick **No, keep planning** until you have reviewed the plan.

![Claude Code editor GUI showing the returned inspection plan and independent scientific expectations](../book/screenshots/claude-editor-plan.svg)

**Claude Code plan:** **1** checks a hand-calculated unequal-error example, **2** lists array-shape and positive-error validation, and **3** confirms Plan mode remained active ([full-size image](../book/screenshots/claude-editor-plan.svg)).
:::
::::

> Read TASK.md, weighted_mean.py and test_weighted_mean.py in this workspace. Explain why pytest currently fails and propose the smallest repair plan. Return one scalar inverse-variance weighted mean, list valid/invalid input assumptions, and use the Course Python interpreter recorded in TASK.md. Do not edit files, install packages, commit or run network commands. Stop after the plan.

**Critique the plan before proceeding:** Did it actually read the files, preserve a **scalar** return rather than `(mean, uncertainty)`, specify $w_i = 1/\sigma_i^2$ rather than $1/\sigma_i$, and keep scope inside `outputs/agent-practice`?


## Own the expected answers; delegate the test syntax

For repeated measurements $x_i$ of the **same quantity** with independent Gaussian one-sigma uncertainties $\sigma_i$, minimising $\sum_i (x_i - \mu)^2 / \sigma_i^2$ gives

$$\hat{\mu} = \frac{\sum_i w_i x_i}{\sum_i w_i}, \qquad w_i = \frac{1}{\sigma_i^2}$$

(its standard error $(\sum_i w_i)^{-1/2}$ is omitted by this exercise's scalar contract). For `values = [1, 3]` and `errors = [1, 3]`, $(1 + 3/9)/(1 + 1/9) = 1.2$; reversing errors gives $2.8$, equal errors give $2.0$, rescaling every $\sigma_i$ by a positive constant leaves $\hat{\mu}$ unchanged, and zero, negative or non-finite uncertainties are invalid.

For observers, inverse-variance weights $w_i = 1/\sigma_i^2$ combine multi-epoch flux/velocity measurements optimally under Gaussian errors; for theorists, $w_i = 1/\sigma_i$ passes equal-error and scale-invariance checks yet fails physically because it **violates Fisher-information additivity** when pooling sub-samples and suppresses noisy $\sigma_2 \gg \sigma_1$ outliers too slowly ($1/\sigma$ vs $1/\sigma^2$).

Run the cell below to see a plausible-looking $1/\sigma$ proposal fail on unequal uncertainties:


In [ ]:
import numpy as np

def candidate_weighted_mean(values, errors):
    # Plausible-looking bug: weights by 1/sigma instead of 1/sigma**2.
    weights = 1 / np.asarray(errors, dtype=float)
    return np.average(values, weights=weights)

candidate = candidate_weighted_mean([1, 3], [1, 3])
print('Candidate:', candidate, '| independent expected answer: 1.2')
assert not np.isclose(candidate, 1.2)

### Task — derive expectations and checkpoint tests

On paper, derive one **new unequal-error case** of your own, plus one **equal-error case** and a **matching 2-D input** (such as `[[1, 3]]`) that must raise `ValueError`. Switch your agent to controlled editing (or write `test_student_checks.py` directly on the manual route) to translate *your* numbers into `pytest` syntax:

::::{tab-set}
:::{tab-item} Jupyter AI on CPD
:sync: agent-jupyterai

In the same **OpenCode** chat, switch to **build** (controlled permissions) and send the test-only prompt. Choose **Allow once** only for writing `test_student_checks.py` and running `pytest` with `/opt/course/.venv/bin/python`; **Reject** any edit to `weighted_mean.py`.

![CPD Jupyter AI with an OpenCode proposal to create the named student test file, independently derived assertions and one-time approval controls](../book/screenshots/jupyter-ai-review.svg)

**Jupyter AI test review:** **1** names `test_student_checks.py`, **2** shows the hand-derived assertions, **3** presents Allow once / Reject, and **4** confirms OpenCode + CPD Qwen ([full-size image](../book/screenshots/jupyter-ai-review.svg)).
:::

:::{tab-item} Cline + CPD Qwen
:sync: agent-cline

Switch **Cline** to **Act** while keeping **Auto-approve: None**. Send the test-only prompt once, read every assertion in the proposed `test_student_checks.py` diff before clicking **Save**, and approve the focused `pytest` command separately.

![Cline with CPD Qwen showing the proposed student test file and pending Save or Reject controls](../book/screenshots/cline-tests.svg)

**Cline test review:** **1** holds Save/Reject with Auto-approve: None, **2** checks the hand-derived `3.6` case, **3** asserts `ValueError` on 2-D arrays, and **4** shows CPD Qwen ([full-size image](../book/screenshots/cline-tests.svg)).
:::

:::{tab-item} Claude Code
:sync: agent-claude

Switch **Claude Code** from Plan to **Manual** and send the test-only prompt. Inspect the proposed `test_student_checks.py` diff and click one-time **Yes** (avoid **Allow all edits this session** so later writes still pause for review).

![Claude Code GUI showing the proposed independent test file and pending one-time write approval](../book/screenshots/claude-tests.svg)

**Claude Code test review:** **1** shows `3.6`, equal-error and 2-D `ValueError` checks, **2** grants one-time Yes, and **3** marks the broader session-wide toggle to avoid ([full-size image](../book/screenshots/claude-tests.svg)).
:::
::::

> Create only test_student_checks.py. Translate my independently calculated cases into pytest tests: [insert your inputs, expected scalar answers and 2-D ValueError case]. Import weighted_mean from weighted_mean, and use pytest.approx and pytest.raises(ValueError). Do not derive expectations from weighted_mean.py, edit test_weighted_mean.py or repair weighted_mean.py. Run pytest -q in this folder with the Course Python interpreter in TASK.md (failures are expected) and stop for review.

Verify every assertion against your notes, confirm the expected baseline failures in the workspace terminal, and lock your **tests checkpoint** (`TESTS_COMMIT`):

```bash
python -m pytest -q
git add test_student_checks.py
git commit -m "Add student weighted-mean checks"
TESTS_COMMIT=$(git rev-parse --short HEAD)
echo "Tests checkpoint: $TESTS_COMMIT"
```


## Authorise a repair and steer the agent

Keep controlled approvals active and request a single-file repair against your locked tests (or implement the fix manually and inspect `git diff` before running `pytest`):

> Repair only weighted_mean.py to satisfy TASK.md and our committed tests. Accept matching nonempty finite 1-D arrays with positive one-sigma errors and return one scalar inverse-variance weighted mean. Read test_weighted_mean.py and test_student_checks.py without editing them or weakening assertions. Run pytest -q in this workspace using the Course Python interpreter in TASK.md, show the diff, and state the statistical assumptions. Do not install packages, use network commands, commit or push.

![Claude Code showing a pending one-file edit diff and approval choice on weighted_mean.py](../book/screenshots/claude-repair-approval.svg)

**Inspect the single-file diff before approving:** **1** confirms only `weighted_mean.py` is targeted, **2** shows inverse-variance weights and a scalar `float` return, and **3** approves just this one edit ([full-size image](../book/screenshots/claude-repair-approval.svg)).

If the agent picks the wrong Python binary, stalls, or touches a test file, steer with concrete evidence and scope rather than vague retries:

- **Bad steering:** `"try again"`
- **Good steering:** `"pytest failed with ModuleNotFoundError under /usr/bin/python3. Re-run with the Course Python interpreter in TASK.md; edit only weighted_mean.py; keep test files at TESTS_COMMIT."`

If an agent modifies either test file, restore your checkpoint immediately before continuing:

```bash
git checkout "$TESTS_COMMIT" -- test_weighted_mean.py test_student_checks.py
```

Note one concrete approval, diff check or steering decision you made during the repair.


## Review the change independently

Never stop at the agent's summary banner. Run the independent verification commands yourself in the `outputs/agent-practice` terminal (setting `TESTS_COMMIT` to your recorded hash if you opened a new shell):

```bash
pytest -q
git status --short
git diff "$TESTS_COMMIT" -- test_weighted_mean.py test_student_checks.py
git diff main -- weighted_mean.py
```

Confirm all tests pass, `git status --short` shows no stray untracked files, the test-file diff against `$TESTS_COMMIT` is **empty**, and `weighted_mean.py` validates shapes/finiteness/positivity while computing $w_i \propto 1/\sigma_i^2$.

::::{tab-set}
:::{tab-item} Jupyter AI on CPD
:sync: agent-jupyterai

Expand the OpenCode edit and `pytest` tool cards to verify what actually ran, inspect the side-by-side diff in JupyterLab's **Git** sidebar, and rerun `pytest -q` in the workspace terminal.

![CPD JupyterLab showing the original saved Jupyter AI conversation beside the native Git repair diff](../book/screenshots/jupyter-ai-final-review.svg)

**Jupyter AI final review:** **1** shows the native Git diff on `weighted_mean.py`, **2** verifies inverse-variance weights, **3** shows the chat's `12 passed` report, and **4** identifies OpenCode + CPD Qwen ([full-size image](../book/screenshots/jupyter-ai-final-review.svg)).
:::

:::{tab-item} Cline + CPD Qwen
:sync: agent-cline

Expand Cline's diff and `pytest` output card (checking the exit code if a banner looks ambiguous), inspect VS Code Source Control against `$TESTS_COMMIT`, and rerun `pytest -q` in your terminal.

![Cline with CPD Qwen showing the repair diff and test evidence to check independently](../book/screenshots/cline-review.svg)

**Cline final review:** **1** scopes the diff to `weighted_mean.py`, **2** shows inverse-variance weighting, **3** records `12 passed` (`PYTEST_EXIT=0`), and **4** confirms CPD Qwen ([full-size image](../book/screenshots/cline-review.svg)).
:::

:::{tab-item} Claude Code
:sync: agent-claude

Inspect the working-tree diff in VS Code and expand Claude Code's executed `pytest` card, then verify unchanged tests against `$TESTS_COMMIT` and rerun `pytest -q` in your own terminal.

![Claude Code editor GUI showing the repair input-validation diff, weighted-mean formula and actual test output](../book/screenshots/claude-review.svg)

**Claude Code final review:** **1** shows validation guards in `weighted_mean.py`, **2** confirms inverse-variance scalar weighting, and **3** displays the executed `12 passed` output ([full-size image](../book/screenshots/claude-review.svg)).
:::
::::

**Review check:** Check that you can explain why you must reject (a) a proposal that edits `1.2` to `1.5` in the test file so $1/\sigma$ weights pass, and (b) a proposal that returns `(mean, uncertainty)` like `course2026.analysis.weighted_mean`.


## Accept the work and remain accountable

Commit your verified `weighted_mean.py` repair on `agent-repair`, then save a concise `AI_USE.md` in `outputs/agent-practice` using this 5-line template:

```markdown
- **Route & host:** [Jupyter AI on CPD / Cline + CPD Qwen / Claude Code / Manual] | Model: [...] | Commands ran on: [CPD / laptop]
- **Scope & checkpoint:** Repair `weighted_mean.py` only | `TESTS_COMMIT`: [short hash]
- **Steering / review decision:** [One concrete approval, path correction or scope boundary you enforced]
- **Observed verification:** Hand-derived case `[...] -> ...` | Before: `[X failed]` | After: `[Y passed]`, test diff vs `TESTS_COMMIT` empty
- **Untested scientific assumption:** [e.g. observations estimate one common value with independent Gaussian errors]
```

**Exit check:** Distinguish the model, agent client, OS sandbox and Git checkpoint; show your clean diff and independent `pytest` run; and state one physical assumption that unit tests cannot prove. For thesis and publication work, agree AI tool use with your supervisor and consult the Portsmouth [Research Degrees Operational Handbook](https://guidelines.docstore.port.ac.uk/ResearchDegreeOperationalHandbook.pdf).


## Optional follow-on: parameter recovery and survey selection

This optional extension applies the same independent-verification habit to a weighted linear fit $v = a d + b$ on Session 1's synthetic `mock_galaxies.ecsv` catalogue (distances in Mpc treated as exact; velocity uncertainties in km/s treated as known independent Gaussian standard deviations via SciPy `sigma` with `absolute_sigma=True`). Before running it, predict what doubling every error bar will do to the best-fit parameters, their standard errors and reduced $\chi^2$.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import QTable
# The Hub pulls the course into your workspace before starting this notebook.
# Prefer that editable checkout; the local editable install is a fallback.
import sys
_candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / "jupyter_data_languages"]
for _candidate in _candidates:
    if (_candidate / "course2026/__init__.py").is_file() and (_candidate / "data/mock_galaxies.ecsv").is_file():
        sys.path.insert(0, str(_candidate))
        break
import course2026
ROOT = Path(course2026.__file__).resolve().parents[1]
plt.rcParams.update({'figure.figsize': (7, 4), 'font.size': 11})

In [ ]:
from course2026.analysis import usable_rows, fit_velocity, velocity_model
table = QTable.read(ROOT / 'data/mock_galaxies.ecsv')
valid = usable_rows(table)
p, covariance = fit_velocity(table['distance'][valid], table['velocity'][valid],
                             table['velocity_error'][valid])
parameter_error = np.sqrt(np.diag(covariance))
truth = np.array([table.meta['slope'], table.meta['intercept']])
print('Fitted [slope, intercept]:', p.round(2))
print('Standard errors:', parameter_error.round(2))
print('Injected truth:', truth)
print('Difference / standard error:', ((p - truth) / parameter_error).round(2))
assert np.all(np.abs(p - truth) < 4 * parameter_error)

In [ ]:
x = table['distance'][valid].to_value(u.Mpc)
y = table['velocity'][valid].to_value(u.km / u.s)
sigma = table['velocity_error'][valid].to_value(u.km / u.s)
residual = (y - velocity_model(x, *p)) / sigma
reduced_chi2 = np.sum(residual ** 2) / (len(x) - 2)
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 6), height_ratios=[2, 1])
axes[0].errorbar(x, y, yerr=sigma, fmt='.', alpha=0.3)
grid = np.linspace(x.min(), x.max(), 100)
axes[0].plot(grid, velocity_model(grid, *p), color='black', label='Weighted fit')
axes[0].set(ylabel='Velocity [km/s]', title='Synthetic parameter recovery')
axes[0].legend()
axes[1].plot(x, residual, '.', alpha=0.5)
axes[1].axhline(0, color='black')
axes[1].set(xlabel='Distance (assumed exact) [Mpc]', ylabel='Residual / sigma')
fig.tight_layout()
plt.show()
print('Reduced chi-squared:', round(reduced_chi2, 2))

### Check uncertainty scaling and survey selection

Comparing `p` against `truth` is only possible in a simulation, and one random seed checks consistency rather than frequentist coverage. Run the cell below to verify how `2 * sigma` rescales parameter uncertainties and reduced $\chi^2$.

**Bridge to Session 1:** Re-fit on `selected = valid & (table['apparent_magnitude'] < 16.5)` to verify how a flux limit removes distant galaxies, shrinks the distance lever arm, and widens the slope uncertainty $\sigma_a$ without biasing the toy slope!


In [ ]:
p2, cov2 = fit_velocity(table['distance'][valid], table['velocity'][valid],
                       2 * table['velocity_error'][valid])
np.testing.assert_allclose(p2, p, rtol=1e-6)
np.testing.assert_allclose(np.sqrt(np.diag(cov2)), 2 * parameter_error, rtol=1e-6)
residual2 = y - velocity_model(x, *p2)
reduced_chi2_2 = np.sum((residual2 / (2 * sigma)) ** 2) / (len(x) - 2)
np.testing.assert_allclose(reduced_chi2_2, reduced_chi2 / 4, rtol=1e-6)
print('Recomputed reduced chi-squared with twice the errors:', round(reduced_chi2_2, 2))
